In [ ]:
import copy
import os.path as osp

import numpy as np
import pandas as pd
from calisim.data_model import (
	DistributionModel,
	ParameterDataType,
	ParameterSpecification,
)
from calisim.sensitivity import (
	SensitivityAnalysisMethod,
	SensitivityAnalysisMethodModel,
)
from pcse.base import ParameterProvider
from pcse.input import (
	DummySoilDataProvider,
	NASAPowerWeatherDataProvider,
	WOFOST72SiteDataProvider,
	YAMLAgroManagementReader,
	YAMLCropDataProvider,
)
from pcse.models import Wofost72_PP

In [ ]:
netherlands_wdp = NASAPowerWeatherDataProvider(latitude=51, longitude=5)
print(netherlands_wdp)

In [ ]:
india_wdp = NASAPowerWeatherDataProvider(latitude=23, longitude=73)
print(india_wdp)

In [ ]:
sited = WOFOST72SiteDataProvider(WAV=50)
print(sited)

In [ ]:
soild = DummySoilDataProvider()
print(soild)

In [ ]:
cropd = YAMLCropDataProvider(fpath=osp.join("..", "data"), force_reload=True)
print(cropd)

In [ ]:
netherlands_agro = YAMLAgroManagementReader(osp.join("..", "data", "potato_netherlands_2021.agro"))
print(netherlands_agro)

In [ ]:
india_agro = YAMLAgroManagementReader(osp.join("..", "data", "potato_india_2021.agro"))
print(india_agro)

In [ ]:
params = ParameterProvider(cropdata=cropd, sitedata=sited, soildata=soild)
wofost = Wofost72_PP(params, netherlands_wdp, netherlands_agro)
ground_truth = dict(
    TSUM1=255,
    TSUM2=1400,
    TBASEM=3.0,
    TSUMEM=170.0,
    TEFFMX=18.0,
    SPAN=37,
    TDWI=75,
    RGRLAI=0.016,
    Q10=2.0
)
for k in ground_truth:
    params.set_override(k, ground_truth[k])
wofost.run_till_terminate()
observed_data = pd.DataFrame(wofost.get_output())
observed_data

In [ ]:
observed_data.plot.scatter(x="day", y="LAI")

In [ ]:
observed_data.plot.scatter(x="day", y="DVS")

In [ ]:
observed_data.plot.scatter(x="day", y="TWSO")

In [ ]:
parameter_spec = ParameterSpecification(
	parameters=[
		DistributionModel(
			name="TSUM1",
			distribution_name="uniform",
			distribution_args=[150, 280],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="TSUM2",
			distribution_name="uniform",
			distribution_args=[1550, 2100],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="TBASEM",
			distribution_name="uniform",
			distribution_args=[2, 4],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="TSUMEM",
			distribution_name="uniform",
			distribution_args=[170, 255],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="TEFFMX",
			distribution_name="uniform",
			distribution_args=[18, 32],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="SPAN",
			distribution_name="uniform",
			distribution_args=[20, 50],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="TDWI",
			distribution_name="uniform",
			distribution_args=[75, 700],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="RGRLAI",
			distribution_name="uniform",
			distribution_args=[0.008, 0.02],
			data_type=ParameterDataType.CONTINUOUS,
		),
		DistributionModel(
			name="Q10",
			distribution_name="uniform",
			distribution_args=[2, 3],
			data_type=ParameterDataType.CONTINUOUS,
		),
	]
)

In [ ]:
state_vars = ["LAI", "TWSO"]

def sensitivity_func(
	parameters: dict, simulation_id: str, observed_data: np.ndarray | None, wdp: NASAPowerWeatherDataProvider, agro: list[dict]
) -> float | list[float]:
    p = copy.deepcopy(params)
    for k in parameters:
        p.set_override(k, parameters[k])

    wofost = Wofost72_PP(p, wdp, agro)
    wofost.run_till_terminate()
    simulated_data = pd.DataFrame(wofost.get_output())
    end_of_season = simulated_data.tail(1)
    return end_of_season[state_vars].values.flatten()

In [ ]:
specification = SensitivityAnalysisMethodModel(
	experiment_name="netherlands_sensitivity_analysis",
	parameter_spec=parameter_spec,
	observed_data=observed_data.LAI.values,
	method="sobol",
	n_samples=256,
	n_jobs=10,
	output_labels=state_vars,
	verbose=True,
	batched=False,
	method_kwargs=dict(calc_second_order=False, scramble=True),
	analyze_kwargs=dict(
		calc_second_order=False,
		num_resamples=200,
		conf_level=0.95,
	),
    calibration_func_kwargs=dict(wdp=netherlands_wdp, agro=netherlands_agro),
)

calibrator = SensitivityAnalysisMethod(
	calibration_func=sensitivity_func, specification=specification, engine="salib"
)

calibrator.specify().execute().analyze()

In [ ]:
calibrator.implementation.sp.to_df()

In [ ]:
specification = SensitivityAnalysisMethodModel(
	experiment_name="india_sensitivity_analysis",
	parameter_spec=parameter_spec,
	observed_data=observed_data.LAI.values,
	method="sobol",
	n_samples=256,
	n_jobs=10,
	output_labels=state_vars,
	verbose=True,
	batched=False,
	method_kwargs=dict(calc_second_order=False, scramble=True),
	analyze_kwargs=dict(
		calc_second_order=False,
		num_resamples=200,
		conf_level=0.95,
	),
    calibration_func_kwargs=dict(wdp=india_wdp, agro=india_agro),
)

calibrator = SensitivityAnalysisMethod(
	calibration_func=sensitivity_func, specification=specification, engine="salib"
)

calibrator.specify().execute().analyze()

In [ ]:
calibrator.implementation.sp.to_df()

In [ ]:
calibrator.implementation.sp.to_df()[0][1]